## 1. Setup & Installation

In [1]:
# Install additional packages for evaluation
%pip install scikit-learn pandas matplotlib seaborn tqdm statsmodels


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import os
import torch
import numpy as np
import pandas as pd
import json
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Create directories
os.makedirs('./data', exist_ok=True)
os.makedirs('./evaluation_results', exist_ok=True)
os.makedirs('./evaluation_results/figures', exist_ok=True)

# Device setup
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print("Using MPS (Apple Silicon)")
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device('cpu')
    print("Using CPU")

print(f"Device: {DEVICE}")

Using MPS (Apple Silicon)
Device: mps


In [3]:
# Import evaluation framework
from evaluation_framework import (
    ASAGSample,
    SemEvalDataLoader,
    BaselineModels,
    EvaluationMetrics,
    KFoldEvaluator,
    AblationStudy,
    ResultVisualizer,
    ResultExporter
)

print("Evaluation framework loaded successfully!")

Evaluation framework loaded successfully!


## 2. Load Dataset

Using synthetic data in SemEval-2013 format. For your actual research, replace with:
- SemEval-2013 Task 7 (Scientsbank + Beetle)
- Mohler et al. Dataset
- ASAP-SAS Dataset

In [4]:
# ============================================================
# CONFIGURATION - Adjust for faster/slower evaluation
# ============================================================
FAST_MODE = True  # Set to False for full evaluation (takes much longer)

if FAST_MODE:
    N_SAMPLES = 150      # Reduced samples for quick testing
    N_FOLDS = 3          # Reduced folds
else:
    N_SAMPLES = 600      # Full samples for paper
    N_FOLDS = 5          # Standard 5-fold

print(f"Mode: {'FAST (testing)' if FAST_MODE else 'FULL (paper)'}")
print(f"Samples: {N_SAMPLES}, Folds: {N_FOLDS}")

# Initialize data loader
data_loader = SemEvalDataLoader()

# Create synthetic dataset
samples = data_loader.create_synthetic_dataset(n_samples=N_SAMPLES)

# Save for reproducibility
data_loader.save_to_json('./data/asag_evaluation_dataset.json', samples)

# Dataset statistics
from collections import Counter
label_dist = Counter([s.gold_label for s in samples])
print(f"\nDataset Statistics:")
print(f"  Total samples: {len(samples)}")
print(f"  Label distribution:")
for label, count in sorted(label_dist.items()):
    print(f"    - {label}: {count} ({count/len(samples)*100:.1f}%)")

Creating synthetic dataset with 600 samples...
Created 600 synthetic samples
Label distribution: {'partially_correct_incomplete': 200, 'correct': 200, 'contradictory': 200}
Saved 600 samples to ./data/asag_evaluation_dataset.json

Dataset Statistics:
  Total samples: 600
  Label distribution:
    - contradictory: 200 (33.3%)
    - correct: 200 (33.3%)
    - partially_correct_incomplete: 200 (33.3%)


In [5]:
# Preview samples
print("\nSample Preview:")
print("="*70)
for i, sample in enumerate(samples[:3]):
    print(f"\nSample {i+1} (Label: {sample.gold_label})")
    print(f"  Q: {sample.question[:80]}...")
    print(f"  Ref: {sample.reference_answer[:80]}...")
    print(f"  Stu: {sample.student_answer[:80]}...")


Sample Preview:

Sample 1 (Label: partially_correct_incomplete)
  Q: Describe how a simple electric circuit works....
  Ref: A simple electric circuit consists of a power source, conducting wires, and a lo...
  Stu: Electricity flow through wires to power things. But I'm not sure about details....

Sample 2 (Label: correct)
  Q: What is mitosis and what is its purpose?...
  Ref: Mitosis is a form of cell division where a single cell divides to produce two id...
  Stu: Mitosis is cell division that creates two identical daughter cells with the same...

Sample 3 (Label: correct)
  Q: Explain Darwin's theory of natural selection....
  Ref: Natural selection is a mechanism of evolution where individuals with favorable t...
  Stu: Natural selection means organisms with advantageous traits survive and reproduce...


## 3. Initialize Models

Loading all required models:
- Hybrid ASAG Grader (your model)
- Baseline models for comparison

In [6]:
# Install nbformat to run notebooks from notebooks
%pip install nbformat --quiet

# Load your Hybrid ASAG Grader
# Import from your main notebook or module
%run hybrid_asag_grader.ipynb

# Initialize grader
grader = HybridASAGGrader(device=DEVICE, verbose=True)


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@

No sentence-transformers model found with name princeton-nlp/sup-simcse-roberta-large. Creating a new one with mean pooling.


      Done: princeton-nlp/sup-simcse-roberta-large

[2/6] Loading KeyBERT Model...
      Done: all-MiniLM-L6-v2 (KeyBERT backend)

[3/6] Loading Formality Model...
      Done: cointegrated/roberta-base-formality

[4/6] Loading Grammar Model...


Some weights of the model checkpoint at textattack/roberta-base-CoLA were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


      Done: textattack/roberta-base-CoLA

[5/6] Loading Logic Model (NLI)...
      Done: cross-encoder/nli-deberta-v3-base

[6/6] Loading Reasoning Model (Qwen2.5-3B-Instruct)...


...

      Done: princeton-nlp/sup-simcse-roberta-large

[2/6] Loading KeyBERT Model...
      Done: all-MiniLM-L6-v2 (KeyBERT backend)

[3/6] Loading Formality Model...
      Done: cointegrated/roberta-base-formality

[4/6] Loading Grammar Model...


Some weights of the model checkpoint at textattack/roberta-base-CoLA were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


      Done: textattack/roberta-base-CoLA

[5/6] Loading Logic Model (NLI)...
      Done: cross-encoder/nli-deberta-v3-base

[6/6] Loading Reasoning Model (Qwen2.5-3B-Instruct)...


Loading checkpoint shards: 100%|██████████| 2/2 [00:15<00:00,  7.94s/it]


      Done: Qwen/Qwen2.5-3B-Instruct

All models loaded successfully!



In [7]:
# Initialize baseline models
baselines = BaselineModels(device=DEVICE)

# Initialize K-Fold evaluator
kfold = KFoldEvaluator(n_splits=5, random_state=42)

print("All models initialized!")

All models initialized!


## 4. Define Grading Functions

Create wrapper functions that take samples and return predictions.

In [8]:
def hybrid_grader_function(samples):
    """
    Wrapper for Hybrid ASAG Grader.
    Returns predictions in SemEval label format.
    """
    # Label mapping from Hybrid to SemEval format
    label_map = {
        'Correct': 'correct',
        'Partially Correct': 'partially_correct_incomplete',
        'Incorrect': 'contradictory'
    }
    
    predictions = []
    
    for sample in tqdm(samples, desc="Hybrid ASAG"):
        result = grader.grade_answer(
            context=sample.context,
            question=sample.question,
            reference=sample.reference_answer,
            student=sample.student_answer
        )
        
        # Get correctness tag
        if result.feedback.tags:
            tag = result.feedback.tags[0]
            pred = label_map.get(tag, 'contradictory')
        else:
            pred = 'contradictory'
        
        predictions.append(pred)
    
    return predictions


def hybrid_rule_only_function(samples):
    """
    Hybrid model without LLM (rule-based only).
    Uses pre-computed metrics and thresholds.
    """
    predictions = []
    
    for sample in tqdm(samples, desc="Hybrid (Rule-Only)"):
        # Get metrics
        semantic = grader.get_semantic_score(sample.reference_answer, sample.student_answer)
        coverage, missing = grader.get_keyword_coverage(
            sample.question, sample.reference_answer, sample.student_answer
        )
        grammar = grader.get_grammar_score(sample.student_answer)
        logic, details = grader.get_logic_score(
            sample.reference_answer, sample.student_answer, grammar
        )
        
        # Weighted composite
        composite = semantic * 0.45 + coverage * 0.20 + logic * 0.20 + grammar * 0.15
        
        # Check contradiction
        max_contradiction = max(
            details.get('contradiction', 0),
            details.get('backward_contradiction', 0)
        )
        
        if max_contradiction > 0.6:
            pred = 'contradictory'
        elif composite >= 0.75:
            pred = 'correct'
        elif composite >= 0.45:
            pred = 'partially_correct_incomplete'
        else:
            pred = 'contradictory'
        
        predictions.append(pred)
    
    return predictions

print("Grading functions defined!")

Grading functions defined!


## 5. Run Baseline Evaluations (5-Fold CV)

In [9]:
# Store all results
all_results = {}

# ============================================================
# BASELINE 1: TF-IDF + Cosine Similarity
# ============================================================
print("\n" + "="*70)
print("BASELINE 1: TF-IDF + Cosine Similarity")
print("="*70)

tfidf_result = kfold.evaluate_model(
    samples,
    baselines.baseline_tfidf,
    "TF-IDF"
)
all_results['TF-IDF'] = tfidf_result


BASELINE 1: TF-IDF + Cosine Similarity

K-Fold Cross Validation: TF-IDF

Fold 1/5
----------------------------------------


TF-IDF Baseline: 100%|██████████| 120/120 [00:00<00:00, 912.49it/s]


  Accuracy: 0.3333
  Macro F1: 0.1667
  QWK:      0.0000

Fold 2/5
----------------------------------------


TF-IDF Baseline: 100%|██████████| 120/120 [00:00<00:00, 1943.67it/s]


  Accuracy: 0.3333
  Macro F1: 0.1667
  QWK:      0.0000

Fold 3/5
----------------------------------------


...

  Accuracy: 0.3333
  Macro F1: 0.1667
  QWK:      0.0000

Fold 4/5
----------------------------------------


TF-IDF Baseline: 100%|██████████| 120/120 [00:00<00:00, 2085.15it/s]


  Accuracy: 0.3333
  Macro F1: 0.1667
  QWK:      0.0000

Fold 5/5
----------------------------------------


TF-IDF Baseline: 100%|██████████| 120/120 [00:00<00:00, 2053.91it/s]


  Accuracy: 0.3333
  Macro F1: 0.1667
  QWK:      0.0000

Aggregated Results (5-Fold)
  Accuracy:    0.3333 ± 0.0000
  Macro F1:    0.1667 ± 0.0000
  Weighted F1: 0.1667 ± 0.0000
  QWK:         0.0000 ± 0.0000
  Pearson:     nan ± nan
  Spearman:    nan ± nan


In [10]:
# ============================================================
# BASELINE 2: SBERT Semantic Only
# ============================================================
print("\n" + "="*70)
print("BASELINE 2: SBERT Semantic Similarity")
print("="*70)

sbert_result = kfold.evaluate_model(
    samples,
    baselines.baseline_sbert_semantic,
    "SBERT"
)
all_results['SBERT'] = sbert_result


BASELINE 2: SBERT Semantic Similarity

K-Fold Cross Validation: SBERT

Fold 1/5
----------------------------------------
Loaded SBERT model: all-MiniLM-L6-v2


SBERT Baseline: 100%|██████████| 120/120 [00:02<00:00, 45.92it/s] 


  Accuracy: 0.4833
  Macro F1: 0.4977
  QWK:      0.5156

Fold 2/5
----------------------------------------


SBERT Baseline: 100%|██████████| 120/120 [00:00<00:00, 179.26it/s]


  Accuracy: 0.4917
  Macro F1: 0.5158
  QWK:      0.5850

Fold 3/5
----------------------------------------


...

  Accuracy: 0.4333
  Macro F1: 0.4623
  QWK:      0.5211

Fold 4/5
----------------------------------------


SBERT Baseline: 100%|██████████| 120/120 [00:00<00:00, 182.21it/s]


  Accuracy: 0.4417
  Macro F1: 0.4656
  QWK:      0.4962

Fold 5/5
----------------------------------------


SBERT Baseline: 100%|██████████| 120/120 [00:00<00:00, 167.31it/s]

  Accuracy: 0.4583
  Macro F1: 0.4755
  QWK:      0.5113

Aggregated Results (5-Fold)
  Accuracy:    0.4617 ± 0.0227
  Macro F1:    0.4834 ± 0.0204
  Weighted F1: 0.4834 ± 0.0204
  QWK:         0.5259 ± 0.0307
  Pearson:     0.5385 ± 0.0276
  Spearman:    0.5277 ± 0.0263


In [11]:
# ============================================================
# BASELINE 3: Rule-based Keyword Matching
# ============================================================
print("\n" + "="*70)
print("BASELINE 3: Keyword Matching")
print("="*70)

keyword_result = kfold.evaluate_model(
    samples,
    baselines.baseline_keyword_matching,
    "Keywords"
)
all_results['Keywords'] = keyword_result


BASELINE 3: Keyword Matching

K-Fold Cross Validation: Keywords

Fold 1/5
----------------------------------------


Keyword Baseline: 100%|██████████| 120/120 [00:00<00:00, 46568.88it/s]


  Accuracy: 0.3333
  Macro F1: 0.2204
  QWK:      0.4845

Fold 2/5
----------------------------------------


Keyword Baseline: 100%|██████████| 120/120 [00:00<00:00, 90216.25it/s]


  Accuracy: 0.3333
  Macro F1: 0.2222
  QWK:      0.5000

Fold 3/5
----------------------------------------


...

  Accuracy: 0.3333
  Macro F1: 0.2204
  QWK:      0.4845

Fold 4/5
----------------------------------------


Keyword Baseline: 100%|██████████| 120/120 [00:00<00:00, 104077.02it/s]


  Accuracy: 0.3333
  Macro F1: 0.2204
  QWK:      0.4845

Fold 5/5
----------------------------------------


Keyword Baseline: 100%|██████████| 120/120 [00:00<00:00, 97883.41it/s]

  Accuracy: 0.3333
  Macro F1: 0.2204
  QWK:      0.4845

Aggregated Results (5-Fold)
  Accuracy:    0.3333 ± 0.0000
  Macro F1:    0.2208 ± 0.0007
  Weighted F1: 0.2208 ± 0.0007
  QWK:         0.4876 ± 0.0062
  Pearson:     0.8531 ± 0.0065
  Spearman:    0.8531 ± 0.0065


In [12]:
# ============================================================
# BASELINE 4: LLM Zero-shot (Optional - takes longer)
# ============================================================
# Uncomment to run LLM baseline (requires significant GPU memory)

# print("\n" + "="*70)
# print("BASELINE 4: LLM Zero-shot")
# print("="*70)
# 
# llm_result = kfold.evaluate_model(
#     samples,
#     baselines.baseline_llm_zeroshot,
#     "LLM-ZeroShot"
# )
# all_results['LLM-ZeroShot'] = llm_result

print("Baseline evaluations complete!")

Baseline evaluations complete!


## 6. Evaluate Hybrid ASAG Model

In [13]:
# ============================================================
# HYBRID ASAG (Full Model)
# ============================================================
print("\n" + "="*70)
print("HYBRID ASAG (Full Model)")
print("="*70)

hybrid_result = kfold.evaluate_model(
    samples,
    hybrid_grader_function,
    "Hybrid-ASAG"
)
all_results['Hybrid-ASAG'] = hybrid_result


HYBRID ASAG (Full Model)

K-Fold Cross Validation: Hybrid-ASAG

Fold 1/5
----------------------------------------


Hybrid ASAG:   0%|          | 0/120 [00:00<?, ?it/s]


Calculating metrics...
   - Semantic similarity...
   - Keyword coverage...
   - Formality...
   - Grammar...
   - Logic/coherence...

Generating feedback with Qwen2...


Hybrid ASAG:   1%|          | 1/120 [02:16<4:29:50, 136.05s/it]

Grading complete!


Calculating metrics...
   - Semantic similarity...
   - Keyword coverage...
   - Formality...
   - Grammar...
   - Logic/coherence...

Generating feedback with Qwen2...


...

Grading complete!


Calculating metrics...
   - Semantic similarity...
   - Keyword coverage...
   - Formality...
   - Grammar...
   - Logic/coherence...

Generating feedback with Qwen2...


Hybrid ASAG:  16%|█▌        | 19/120 [35:15<3:01:37, 107.90s/it]

Grading complete!


Calculating metrics...
   - Semantic similarity...
   - Keyword coverage...
   - Formality...
   - Grammar...
   - Logic/coherence...

Generating feedback with Qwen2...


Hybrid ASAG:  16%|█▌        | 19/120 [36:53<3:16:05, 116.49s/it]


KeyboardInterrupt: 

In [14]:
# ============================================================
# HYBRID ASAG (Rule-Based Only - No LLM)
# ============================================================
print("\n" + "="*70)
print("HYBRID ASAG (Rule-Based Only)")
print("="*70)

hybrid_rule_result = kfold.evaluate_model(
    samples,
    hybrid_rule_only_function,
    "Hybrid-RuleOnly"
)
all_results['Hybrid-RuleOnly'] = hybrid_rule_result


HYBRID ASAG (Rule-Based Only)

K-Fold Cross Validation: Hybrid-RuleOnly

Fold 1/5
----------------------------------------


Hybrid (Rule-Only):   0%|          | 0/120 [00:39<?, ?it/s]


KeyboardInterrupt: 

## 7. Results Comparison

In [ ]:
# Create comparison DataFrame
comparison_data = []

for model_name, result in all_results.items():
    row = {
        'Model': model_name,
        'Accuracy': f"{result['accuracy']['mean']:.3f} ± {result['accuracy']['std']:.3f}",
        'Macro F1': f"{result['macro_f1']['mean']:.3f} ± {result['macro_f1']['std']:.3f}",
        'Weighted F1': f"{result['weighted_f1']['mean']:.3f} ± {result['weighted_f1']['std']:.3f}",
        'QWK': f"{result['qwk']['mean']:.3f} ± {result['qwk']['std']:.3f}",
        'Pearson': f"{result['pearson']['mean']:.3f} ± {result['pearson']['std']:.3f}",
        'Spearman': f"{result['spearman']['mean']:.3f} ± {result['spearman']['std']:.3f}"
    }
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*100)
print("MODEL COMPARISON RESULTS (5-Fold Cross Validation)")
print("="*100)
print(comparison_df.to_string(index=False))

In [ ]:
# Visualize comparison
import matplotlib.pyplot as plt
import seaborn as sns

visualizer = ResultVisualizer()

visualizer.plot_model_comparison(
    all_results,
    metrics=['accuracy', 'macro_f1', 'qwk'],
    figsize=(14, 6),
    save_path='./evaluation_results/figures/model_comparison.png'
)

## 8. Statistical Significance Tests

In [ ]:
# Statistical significance: Hybrid vs each baseline
print("\n" + "="*70)
print("STATISTICAL SIGNIFICANCE TESTS")
print("(McNemar's Test: Hybrid-ASAG vs Baselines)")
print("="*70)

gold_labels = [s.gold_label for s in samples]
hybrid_preds = all_results['Hybrid-ASAG']['all_predictions']

significance_results = []

for baseline_name in ['TF-IDF', 'SBERT', 'Keywords', 'Hybrid-RuleOnly']:
    if baseline_name in all_results:
        baseline_preds = all_results[baseline_name]['all_predictions']
        
        try:
            stat, pval, interp = EvaluationMetrics.statistical_significance_test(
                gold_labels, hybrid_preds, baseline_preds, test_type='mcnemar'
            )
            
            significance_results.append({
                'Comparison': f"Hybrid vs {baseline_name}",
                'Statistic': f"{stat:.4f}",
                'p-value': f"{pval:.6f}",
                'Interpretation': interp
            })
            
            print(f"\nHybrid-ASAG vs {baseline_name}:")
            print(f"  Statistic: {stat:.4f}")
            print(f"  p-value:   {pval:.6f}")
            print(f"  Result:    {interp}")
        except Exception as e:
            print(f"  Error: {e}")

sig_df = pd.DataFrame(significance_results)
print("\n" + sig_df.to_string(index=False))

## 9. Ablation Study

In [ ]:
# Initialize ablation study
ablation = AblationStudy(grader)

print("\n" + "="*70)
print("ABLATION STUDY")
print("Testing contribution of each component")
print("="*70)

In [ ]:
# Run ablation study
ablation_results = ablation.run_ablation_study(samples, kfold)

In [ ]:
# Generate ablation table
ablation_df = ablation.generate_ablation_table()

print("\n" + "="*90)
print("ABLATION STUDY RESULTS")
print("="*90)
print(ablation_df.to_string(index=False))

In [ ]:
# Visualize ablation impact
visualizer.plot_ablation_impact(
    ablation_results,
    metric='macro_f1',
    figsize=(12, 8),
    save_path='./evaluation_results/figures/ablation_impact.png'
)

## 10. Confusion Matrices

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Plot confusion matrix for Hybrid ASAG
labels = ['correct', 'partially_correct_incomplete', 'contradictory']
short_labels = ['Correct', 'Partial', 'Incorrect']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models_to_plot = ['TF-IDF', 'SBERT', 'Hybrid-ASAG']

for ax, model_name in zip(axes, models_to_plot):
    if model_name in all_results:
        preds = all_results[model_name]['all_predictions']
        cm = confusion_matrix(gold_labels, preds, labels=labels)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=short_labels, yticklabels=short_labels,
                   ax=ax)
        ax.set_title(model_name)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('./evaluation_results/figures/confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

## 11. Per-Class Analysis

In [ ]:
from sklearn.metrics import classification_report

print("\n" + "="*70)
print("PER-CLASS PERFORMANCE: Hybrid-ASAG")
print("="*70)

if 'Hybrid-ASAG' in all_results:
    hybrid_preds = all_results['Hybrid-ASAG']['all_predictions']
    print(classification_report(gold_labels, hybrid_preds, 
                               target_names=['Correct', 'Partial', 'Incorrect']))

## 12. Export Results for Paper

In [ ]:
# Export all results
exporter = ResultExporter()

# CSV export
exporter.to_csv(all_results, './evaluation_results/main_results.csv')
exporter.to_csv(ablation_results, './evaluation_results/ablation_results.csv')

print("Results exported to CSV!")

In [ ]:
# LaTeX table for paper
latex_main = exporter.to_latex_table(
    all_results,
    metrics=['accuracy', 'macro_f1', 'qwk', 'pearson'],
    caption="Performance comparison of ASAG models (5-fold cross-validation)",
    label="tab:main_results"
)

print("\n" + "="*70)
print("LATEX TABLE (Main Results)")
print("="*70)
print(latex_main)

# Save to file
with open('./evaluation_results/main_results.tex', 'w') as f:
    f.write(latex_main)

In [ ]:
# LaTeX table for ablation
latex_ablation = exporter.to_latex_table(
    ablation_results,
    metrics=['accuracy', 'macro_f1', 'qwk'],
    caption="Ablation study results",
    label="tab:ablation"
)

print("\n" + "="*70)
print("LATEX TABLE (Ablation Study)")
print("="*70)
print(latex_ablation)

# Save to file
with open('./evaluation_results/ablation_results.tex', 'w') as f:
    f.write(latex_ablation)

In [ ]:
# Full JSON export
full_export = {
    'main_results': {
        k: {
            'accuracy': {'mean': v['accuracy']['mean'], 'std': v['accuracy']['std']},
            'macro_f1': {'mean': v['macro_f1']['mean'], 'std': v['macro_f1']['std']},
            'weighted_f1': {'mean': v['weighted_f1']['mean'], 'std': v['weighted_f1']['std']},
            'qwk': {'mean': v['qwk']['mean'], 'std': v['qwk']['std']},
            'pearson': {'mean': v['pearson']['mean'], 'std': v['pearson']['std']},
            'spearman': {'mean': v['spearman']['mean'], 'std': v['spearman']['std']}
        }
        for k, v in all_results.items()
    },
    'ablation_results': {
        k: {
            'accuracy': {'mean': v['accuracy']['mean'], 'std': v['accuracy']['std']},
            'macro_f1': {'mean': v['macro_f1']['mean'], 'std': v['macro_f1']['std']},
            'qwk': {'mean': v['qwk']['mean'], 'std': v['qwk']['std']}
        }
        for k, v in ablation_results.items()
    },
    'statistical_tests': significance_results,
    'dataset_info': {
        'n_samples': len(samples),
        'n_folds': 5,
        'label_distribution': dict(label_dist)
    }
}

with open('./evaluation_results/complete_results.json', 'w') as f:
    json.dump(full_export, f, indent=2)

print("\nComplete results saved to ./evaluation_results/complete_results.json")

## 13. Summary & Conclusions

In [ ]:
print("\n" + "="*70)
print("EVALUATION SUMMARY")
print("="*70)

# Find best model
best_model = max(all_results.items(), key=lambda x: x[1]['macro_f1']['mean'])

print(f"\n📊 Dataset: {len(samples)} samples, 3 classes")
print(f"📈 Evaluation: 5-Fold Stratified Cross-Validation")
print(f"")
print(f"🏆 Best Model: {best_model[0]}")
print(f"   - Accuracy:    {best_model[1]['accuracy']['mean']:.3f} ± {best_model[1]['accuracy']['std']:.3f}")
print(f"   - Macro F1:    {best_model[1]['macro_f1']['mean']:.3f} ± {best_model[1]['macro_f1']['std']:.3f}")
print(f"   - QWK:         {best_model[1]['qwk']['mean']:.3f} ± {best_model[1]['qwk']['std']:.3f}")

# Improvement over baselines
if 'Hybrid-ASAG' in all_results:
    hybrid_f1 = all_results['Hybrid-ASAG']['macro_f1']['mean']
    print(f"\n📈 Improvements over baselines (Macro F1):")
    for baseline in ['TF-IDF', 'SBERT', 'Keywords']:
        if baseline in all_results:
            baseline_f1 = all_results[baseline]['macro_f1']['mean']
            improvement = (hybrid_f1 - baseline_f1) / baseline_f1 * 100
            print(f"   vs {baseline}: +{improvement:.1f}%")

# Ablation insights
print(f"\n🔬 Ablation Study Insights:")
if 'Full Model' in ablation_results:
    full_f1 = ablation_results['Full Model']['macro_f1']['mean']
    for config, result in ablation_results.items():
        if config != 'Full Model':
            drop = full_f1 - result['macro_f1']['mean']
            print(f"   {config}: Δ = {drop:+.3f} F1")

print("\n" + "="*70)
print("Evaluation Complete! All results saved to ./evaluation_results/")
print("="*70)

---

## Files Generated

| File | Description |
|------|-------------|
| `evaluation_results/main_results.csv` | Main comparison results (CSV) |
| `evaluation_results/ablation_results.csv` | Ablation study results (CSV) |
| `evaluation_results/main_results.tex` | LaTeX table for paper |
| `evaluation_results/ablation_results.tex` | LaTeX table for ablation |
| `evaluation_results/complete_results.json` | Full results (JSON) |
| `evaluation_results/figures/model_comparison.png` | Bar chart comparison |
| `evaluation_results/figures/ablation_impact.png` | Ablation impact chart |
| `evaluation_results/figures/confusion_matrices.png` | Confusion matrices |

---

## Next Steps for Research Paper

1. **Replace synthetic data** with real SemEval-2013 or Mohler dataset
2. **Run LLM baseline** (Baseline 4) for complete comparison
3. **Add error analysis** section with representative examples
4. **Run on multiple datasets** for generalizability
5. **Cross-domain evaluation** (train on science, test on other domains)